# Berry Picker Assistant – Yearly harvest labels (Marsi reports)
**Team Picky · ML team · TIES4570 CSDP**

Downloads the official **Marsi** reports (Ruokavirasto, 2013–2025) and extracts **kg of bilberry, lingonberry and cloudberry bought from pickers per area per year**
(areas: Lappi, Oulu-Kainuu, Itä-Suomi, Länsi-Suomi, Koko maa). These are the training labels for the yield model.

- Each report also repeats the previous year → fills missing years (own report always wins).
- Units differ by year (kg vs 1000 kg) → handled in the parser.
- Output: `marsi_labels.csv` (year, area, berry, total_kg, source).

Runs on **CPU** · Internet **On**.

In [ ]:
!pip install -q pdfplumber

In [ ]:
import re, os, requests, pdfplumber, pandas as pd
from urllib.parse import urljoin
LIST = "https://www.ruokavirasto.fi/tuet/elintarvikkeet/metsamarjojen-ja--sienten-varastointituki/marsi-raportit/"
html = requests.get(LIST, timeout=60).text
links = [urljoin(LIST, l) for l in sorted(set(re.findall(r'href="([^"]+\.pdf)"', html)))]
os.makedirs("/tmp/marsi", exist_ok=True)
files = []
for url in links:
    m = re.search(r"(20\d\d)", url)
    if not m: continue
    fn = "/tmp/marsi/" + url.split("/")[-1]
    if not os.path.exists(fn):
        open(fn, "wb").write(requests.get(url, timeout=120).content)
    files.append((int(m.group(1)), fn))
print(len(files), "reports")

In [ ]:
%%writefile /kaggle/working/parse_marsi.py
import re
import itertools

AREA_MAP = {"LAPPI": "Lappi", "OULU": "Oulu-Kainuu", "ITÄ": "Ita-Suomi", "LÄNSI": "Lansi-Suomi", "KOKO": "Koko maa"}
BERRY_MAP = {"Mustikka": "bilberry", "Puolukka": "lingonberry", "Lakka": "cloudberry"}


def area_of(s):
    s = (s or "").upper()
    for k, v in AREA_MAP.items():
        if s.strip().startswith(k):
            return v
    return None


def num(s):
    """'5 511 730' -> 5511730.0 ; '1 038,0' -> 1038.0 ; '...' -> None"""
    if s is None:
        return None
    s = s.strip().replace(" ", " ")
    if not s or set(s) <= set(".…- "):
        return None
    s = s.replace(" ", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def parse_table_rows(rows, report_year):
    """rows: list of cell lists from pdfplumber. Returns list of dicts (block index, area, berry, total_kg)."""
    out, area, block, seen = [], None, 0, set()
    for cells in rows:
        cells = [(c or "").replace("\n", " ").strip() for c in cells]
        if len(cells) < 9:
            continue
        a = area_of(cells[0])
        if a:
            if a in seen:            # same area again -> previous-year block starts
                block += 1
                seen = set()
            seen.add(a)
            area = a
        berry = BERRY_MAP.get(cells[1])
        if not area or not berry:
            continue
        raw = cells[8]
        v = num(raw)
        if v is None:
            continue
        thousands = ("," in raw) or report_year >= 2021
        out.append({"block": block, "area": area, "berry": berry,
                    "total_kg": v * 1000 if thousands else v, "raw": raw})
    return out


def split_kg_eur(tokens, price):
    """tokens like ['1','932','633','2','862','950'] -> pick split where kg*price ~= eur."""
    best = None
    for i in range(1, len(tokens)):
        kg_t, eur_t = tokens[:i], tokens[i:]
        if any(len(t) != 3 for t in kg_t[1:]) or any(len(t) != 3 for t in eur_t[1:]):
            continue
        kg, eur = float("".join(kg_t)), float("".join(eur_t))
        if kg <= 0:
            continue
        err = abs(kg * price - eur) / max(eur, 1)
        if best is None or err < best[0]:
            best = (err, kg, eur)
    return best


def parse_text_line(line):
    """'ITÄ-SUOMI Mustikka 58 946 208 055 3,53 1 551 635 2 228 318 1,44 1 610 581 2 436 373 1,51'"""
    m = re.match(r"^\s*(\S.*?)?\s*(Mustikka|Puolukka|Lakka)\s+(.*)$", line)
    if not m:
        return None
    toks = m.group(3).split()
    prices = [i for i, t in enumerate(toks) if re.fullmatch(r"\d+,\d+", t)]
    if len(prices) < 1:
        return None
    last = prices[-1]
    prev = prices[-2] if len(prices) >= 2 else -1
    group = toks[prev + 1:last]
    group = [t for t in group if re.fullmatch(r"\d+", t)]
    best = split_kg_eur(group, float(toks[last].replace(",", ".")))
    if not best or best[0] > 0.05:
        return None
    return {"area": area_of(m.group(1) or ""), "berry": BERRY_MAP[m.group(2)], "total_kg": best[1]}




In [ ]:
import sys; sys.path.insert(0, "/kaggle/working")
from parse_marsi import parse_table_rows, parse_text_line, area_of

recs = []
for rep_year, fn in files:
    with pdfplumber.open(fn) as pdf:
        for pno, page in enumerate(pdf.pages, 1):
            text = page.extract_text() or ""
            if not re.search(r"Mustikka|Puolukka|Lakka", text):
                continue
            rows = [r for t in (page.extract_tables() or []) for r in t]
            got = parse_table_rows(rows, rep_year)
            src = "table"
            if not got:                                   # fallback: plain text lines (2013-2014 style)
                src, area, block, seen = "line", None, 0, set()
                for line in text.split("\n"):
                    a = area_of(line)
                    r = parse_text_line(line)
                    if a:
                        if a in seen: block += 1; seen = set()
                        seen.add(a); area = a
                    if r:
                        r["area"] = r["area"] or area
                        if r["area"]:
                            r["block"] = block; got.append(r)
            for r in got:
                if r["block"] > 1: continue
                recs.append({"year": rep_year - r["block"], "area": r["area"], "berry": r["berry"],
                             "total_kg": r["total_kg"], "report": rep_year, "page": pno, "src": src,
                             "priority": r["block"]})
R = pd.DataFrame(recs)
print(R.shape)
# keep only blocks that list (almost) all areas -> drops summary pages where national totals were mis-assigned
nareas = R.groupby(["report", "page", "priority"]).area.transform("nunique")
R = R[nareas >= 4]
print("after area-completeness filter:", R.shape)
# keep: own report first (priority 0), then first page found
L = (R.sort_values(["year", "area", "berry", "priority", "page"])
       .drop_duplicates(["year", "area", "berry"], keep="first")
       .reset_index(drop=True))
L.to_csv("/kaggle/working/marsi_labels.csv", index=False)
print(L.shape)

In [ ]:
pd.set_option("display.width", 200)
P = L.pivot_table(index="year", columns=["berry", "area"], values="total_kg", aggfunc="first") / 1000
print("Thousand kg bought from pickers")
P.round(0)

In [ ]:
# Sanity check: areas should add up to roughly 'Koko maa'
chk = L[L.area != "Koko maa"].groupby(["year", "berry"]).total_kg.sum().rename("sum_areas").to_frame()
chk["koko_maa"] = L[L.area == "Koko maa"].set_index(["year", "berry"]).total_kg
chk["ratio"] = (chk.sum_areas / chk.koko_maa).round(2)
chk

# Coverage: which year/berry has how many areas
print(L.groupby(["year", "berry"]).area.nunique().unstack())

## ✅ Done
`marsi_labels.csv` is in the Output tab. Next: join with `weather_features_region_year.csv` and train the yield model.